# Figure 6 | Cedar Rapids geoelectric model and assessment

Run the cells in order from the repository root or `examples/`. Panels are saved to `examples/Figure/fig6/` at 400 dpi. The source field data and saved case results must be obtained separately. The assembled manuscript figure is not generated here.

Ellis CW1 appears only as a limited section comparison; its selected ERT run was vetoed retrospectively. The saved model was produced before that review.

## Setup and saved model inputs

In [ ]:
from pathlib import Path
import json
import os
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, ListedColormap, BoundaryNorm
from matplotlib.cm import ScalarMappable
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator, FuncFormatter
from scipy.spatial import cKDTree
from scipy.interpolate import griddata
from PIL import Image

def find_project_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'examples').is_dir() and (candidate / 'pyproject.toml').is_file():
            return candidate
    raise FileNotFoundError('Run this notebook from the project root or examples directory.')


ROOT = find_project_root()
CASE = ROOT / 'Result' / 'Cedar-Rapids'
OUT = ROOT / 'examples' / 'Figure' / 'fig6'
OUT.mkdir(parents=True, exist_ok=True)

STATE = json.loads((CASE / 'case_state.json').read_text(encoding='utf-8'))
AEM_RUN = CASE / 'runs' / STATE['aem_run_id']
GEO_MODEL = CASE / 'imports' / 'geological_model.npz'

mpl.rcParams.update({
    'font.family': 'Arial', 'font.size': 11,
    'axes.labelsize': 14, 'xtick.labelsize': 11, 'ytick.labelsize': 11,
    'axes.linewidth': 1.1, 'axes.grid': False,
    'legend.frameon': False, 'savefig.facecolor': 'white',
    'figure.facecolor': 'white',
})

RES_CMAP = mpl.colormaps['turbo'].copy()
RES_CMAP.set_bad('white')
RES_NORM = LogNorm(1.0, 1.0e4)
RES_TICKS = [1, 10, 100, 1000, 10000]
RES_TICKLABELS = [r'$10^0$', r'$10^1$', r'$10^2$', r'$10^3$', r'$10^4$']

UNIT_COLORS = np.array([
    mpl.colors.to_rgba('#CDD1D5', 1.00),  # below DOI: visually complete, uninterpreted
    mpl.colors.to_rgba('#E8D8AD', 1.00),  # above rise
    mpl.colors.to_rgba('#D95F47', 1.00),  # resistive domain
    mpl.colors.to_rgba('#4C78A8', 1.00),  # finer fill
])
UNIT_CMAP = ListedColormap(UNIT_COLORS)
UNIT_NORM = BoundaryNorm(np.arange(-0.5, 4.5, 1.0), UNIT_CMAP.N)

def bold_axes(ax):
    ax.xaxis.label.set_fontweight('bold')
    ax.yaxis.label.set_fontweight('bold')
    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontweight('bold')
    ax.tick_params(width=1.2, length=5)

def save_png(fig, filename, *, tight=True):
    path = OUT / filename
    kwargs = {'bbox_inches': 'tight'} if tight else {}
    fig.savefig(path, dpi=400, facecolor='white', **kwargs)
    # Optional vector copy for rendered-text QA; off unless the variable is set.
    qa_dir = os.environ.get('FIG6_QA_PDF_DIR')
    if qa_dir:
        fig.savefig(Path(qa_dir) / Path(filename).with_suffix('.pdf'), facecolor='white', **kwargs)
    plt.close(fig)
    return path

def add_resistivity_colorbar(fig, cax, mappable):
    cb = fig.colorbar(mappable, cax=cax)
    cb.set_ticks(RES_TICKS)
    cb.set_ticklabels(RES_TICKLABELS)
    cb.set_label(r'Resistivity ($\Omega$ m)', fontsize=14, fontweight='bold')
    cb.ax.tick_params(labelsize=11, width=1.2, length=5)
    for label in cb.ax.get_yticklabels():
        label.set_fontweight('bold')
    return cb

def resistivity_rgba(values):
    values = np.asarray(values, float)
    clipped = np.clip(np.nan_to_num(values, nan=1.0), 1.0, 1.0e4)
    return RES_CMAP(RES_NORM(clipped.ravel())).reshape(values.shape + (4,))

def cell_edges(centres):
    centres = np.asarray(centres, dtype=float)
    delta = np.diff(centres)
    return np.r_[centres[0] - delta[0] / 2,
                 (centres[:-1] + centres[1:]) / 2,
                 centres[-1] + delta[-1] / 2]

def style_3d(ax):
    ax.set_proj_type('ortho')
    ax.view_init(elev=25, azim=-58)
    ax.set_box_aspect((2.0, 1.0, 0.72))
    ax.grid(False)
    ax.set_xlabel('Easting (km)', labelpad=20, fontweight='bold')
    ax.set_ylabel('Northing (km)', labelpad=26, fontweight='bold')
    ax.set_zlabel('Depth (m)', labelpad=9, fontweight='bold')
    ax.zaxis.set_major_formatter(FuncFormatter(lambda value, _: f'{value * 1000:.0f}'))
    ax.xaxis.set_major_locator(MaxNLocator(nbins=4))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=4))
    ax.zaxis.set_major_locator(MaxNLocator(nbins=4))
    for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
        axis.pane.fill = False
        axis.pane.set_edgecolor('#B8B8B8')
    for label in ax.get_xticklabels() + ax.get_yticklabels() + ax.get_zticklabels():
        label.set_fontweight('bold')
    ax.tick_params(pad=2)


In [ ]:
aem = np.load(AEM_RUN / 'stitched_conductivity_section.npz')
keep = np.asarray(aem['chi_squared'], float) <= 2.0
aem_xy = np.asarray(aem['receiver_locations_m'][keep, :2], float)
aem_depth_top = np.asarray(aem['layer_top_depth_m'], float)
aem_sigma = np.asarray(aem['conductivity_s_m'][keep], float)
aem_rho = np.divide(1.0, aem_sigma, out=np.full_like(aem_sigma, np.nan), where=aem_sigma > 0)

geo = np.load(GEO_MODEL, allow_pickle=True)
centres = np.asarray(geo['cell_centres_m'], float)
xc = np.unique(centres[:, 0]); yc = np.unique(centres[:, 1])
nx, ny = len(xc), len(yc)
nz = len(centres) // (nx * ny)
column_z = centres.reshape(nx, ny, nz, 3)[:, :, :, 2]
ground = np.asarray(geo['ground_elevation_m'], float).reshape(nx, ny, nz)
depth_grid = ground - column_z
depthc = depth_grid[0, 0]
if not np.allclose(depth_grid, depthc[None, None, :], atol=1e-6):
    raise ValueError('Model columns do not share a regular depth grid')

# The model file is ordered by column, then increasing depth.
unit = np.asarray(geo['unit'], int).reshape(nx, ny, nz)
cover = np.asarray(geo['cover_depth_m'], float).reshape(nx, ny, nz)[:, :, 0]
doi = np.asarray(geo['investigation_depth_m'], float).reshape(nx, ny, nz)[:, :, 0]
support = np.asarray(geo['support_distance_m'], float).reshape(nx, ny, nz)[:, :, 0]

columns = centres.reshape(nx * ny, nz, 3)[:, 0, :2]
distance, nearest = cKDTree(aem_xy).query(columns)
column_distance = distance.reshape(nx, ny)
layer_index = np.searchsorted(aem_depth_top, depthc, side='right') - 1
layer_index = np.clip(layer_index, 0, aem_depth_top.size - 1)
rho_columns = aem_rho[nearest[:, None], layer_index[None, :]]
rho3 = rho_columns.reshape(nx, ny, nz)
valid3 = ((depthc[None, None, :] <= doi[:, :, None]) &
          (column_distance[:, :, None] <= 300.0) & np.isfinite(rho3))
rho3 = np.where(valid3, rho3, np.nan)

xkm = xc / 1000.0; ykm = yc / 1000.0; zkm = depthc / 1000.0
xedges = cell_edges(xkm); yedges = cell_edges(ykm); zedges = cell_edges(zkm)

# Select two orthogonal sections through the central part of the strongest resistive
# body, avoiding edge-driven choices.
y_candidates = np.arange(max(1, ny // 5), min(ny - 1, ny - ny // 5))
x_candidates = np.arange(max(1, nx // 5), min(nx - 1, nx - nx // 5))
iy = int(y_candidates[np.argmax((unit[:, y_candidates, :] == 2).sum(axis=(0, 2)))])
ix = int(x_candidates[np.argmax((unit[x_candidates, :, :] == 2).sum(axis=(1, 2)))])
xcut, ycut = xkm[ix], ykm[iy]

print(f'AEM soundings retained: {keep.sum():,} / {keep.size:,}')
print(f'Volume grid: {nx} x {ny} x {nz} = {nx*ny*nz:,} cells')
print(f'Longitudinal section: N = {ycut:.3f} km')
print(f'Transverse section: E = {xcut:.3f} km')


## a | AEM resistivity

In [ ]:
# Fence diagram: retain the actual stitched 1-D sounding geometry rather than filling
# the 3-D box with interpolated voxels.
fig = plt.figure(figsize=(10.2, 7.6), facecolor='white')
ax = fig.add_subplot(111, projection='3d')
style_3d(ax)

all_ids = np.asarray(aem['sounding_ids'])[keep]
line_id = np.array([name.split('-')[0] for name in all_ids])

# Use the principal E-W production lines and N-S tie lines.
selected_lines = ['L10352', 'L10391', 'L19051']

# DOI at each sounding is inherited from the nearest column of the final model.
column_xy = columns
column_doi = doi.reshape(-1)
sounding_doi = column_doi[cKDTree(column_xy).query(aem_xy)[1]]

# The curtains stop where the model does, at 98 m.
display_depth = np.arange(0.0, zedges[-1] * 1000.0 + 0.1, 2.0)
display_layer = np.clip(
    np.searchsorted(aem_depth_top, display_depth, side='right') - 1,
    0, aem_depth_top.size - 1,
)

for name in selected_lines:
    take = np.flatnonzero(line_id == name)
    points = aem_xy[take]

    # Order each possibly curved flight line by distance along its first principal
    # direction, then split only at genuine acquisition gaps.
    centred = points - points.mean(axis=0)
    direction = np.linalg.svd(centred, full_matrices=False)[2][0]
    along = centred @ direction
    order = np.argsort(along)
    take = take[order]
    points = points[order]
    gaps = np.r_[0, np.cumsum(np.linalg.norm(np.diff(points, axis=0), axis=1) > 110.0)]

    for group in np.unique(gaps):
        local = np.flatnonzero(gaps == group)
        if local.size < 3:
            continue
        index = take[local]
        segment_xy = aem_xy[index]
        segment_rho = aem_rho[index][:, display_layer]
        segment_doi = sounding_doi[index]

        X = np.repeat((segment_xy[:, 0] / 1000.0)[:, None], display_depth.size, axis=1)
        Y = np.repeat((segment_xy[:, 1] / 1000.0)[:, None], display_depth.size, axis=1)
        Z = np.repeat((display_depth / 1000.0)[None, :], local.size, axis=0)
        valid = display_depth[None, :] <= segment_doi[:, None]
        rgba = resistivity_rgba(segment_rho)
        rgba[..., 3] = np.where(valid, 0.96, 0.0)
        ax.plot_surface(X, Y, Z, facecolors=rgba, linewidth=0,
                        antialiased=False, shade=False, rstride=1, cstride=1)
        ax.plot(segment_xy[:, 0] / 1000.0, segment_xy[:, 1] / 1000.0,
                np.zeros(local.size), color='#202020', lw=0.65, alpha=0.85)

ax.set_xlim(xedges[0], xedges[-1])
ax.set_ylim(yedges[0], yedges[-1])
ax.set_zlim(zedges[-1], 0)

# Darken the native 3-D box edges; native panes preserve view-dependent occlusion.
box_color, box_lw = '#111111', 1.15
for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
    axis.pane.set_edgecolor(box_color)
    axis.pane.set_linewidth(box_lw)
    axis.pane.set_alpha(1.0)

# Restore only the observer-facing box edges for this fixed view.
x0, x1 = xedges[0], xedges[-1]
y0, y1 = yedges[0], yedges[-1]
z0, z1 = 0.0, zedges[-1]
front_edges = [
    ([x0, x1], [y0, y0], [z0, z0]),
    ([x1, x1], [y0, y1], [z0, z0]),
    ([x0, x0], [y0, y0], [z0, z1]),
    ([x1, x1], [y0, y0], [z0, z1]),
]
for xe, ye, ze in front_edges:
    ax.plot(xe, ye, ze, color=box_color, lw=box_lw, zorder=100)

cax = fig.add_axes([0.86, 0.22, 0.030, 0.50])
add_resistivity_colorbar(fig, cax, ScalarMappable(norm=RES_NORM, cmap=RES_CMAP))
fig.subplots_adjust(left=0.00, right=0.86, bottom=0.02, top=0.99)
save_png(fig, 'Fig6a_AEM_quasi3D.png')
print('AEM fence lines:', ', '.join(selected_lines))

## b | ERT sections

In [ ]:
ERT_NORM = LogNorm(30.0, 2.0e3)
ERT_TICKS = [30, 100, 1000]
ERT_TICKLABELS = ['30', '100', '1000']

ERT_SITES = ['SVP_S10', 'SVP_CW6', 'Ellis_CW1', 'SVPDemo-Day']
DISPLAY_NAMES = {
    'SVP_S10': 'SVP S10', 'SVP_CW6': 'SVP CW6',
    'Ellis_CW1': 'Ellis CW1', 'SVPDemo-Day': 'SVP Demo-Day',
}

sections = []
for site in ERT_SITES:
    run_id = STATE['ert_runs'][f'{site}/dipole-dipole']
    run = CASE / 'runs' / run_id
    mesh = np.load(run / 'mesh_geometry.npz')
    centres2 = np.asarray(mesh['cell_centers_m'], float)
    model = np.asarray(np.load(run / 'recovered_model.npy'), float)
    item = next(row for row in STATE['ert_imports']
                if row['site'] == site and row['array'] == 'dipole-dipole')
    electrodes = np.asarray(np.load(ROOT / item['electrodes_path']), float)
    span = float(np.ptp(electrodes[:, 0]))
    sections.append((site, span, centres2[:, 0], -centres2[:, 1], model))

# Longest to shortest. Ties retain the specified scientific/site order.
sections = sorted(sections, key=lambda item: -item[1])
max_span = max(item[1] for item in sections)
depth_max = 30.0
frame_w_max = 6.35
frame_h = frame_w_max * depth_max / max_span
left, right = 1.05, 1.78
bottom, top, gap = 0.72, 0.32, 0.36
height = bottom + top + len(sections) * frame_h + (len(sections) - 1) * gap
width = left + frame_w_max + right
fig = plt.figure(figsize=(width, height), facecolor='white')

for row, (site, span, distance_m, depth_m, model) in enumerate(sections):
    frame_w = frame_w_max * span / max_span
    x0 = left + (frame_w_max - frame_w) / 2
    y0 = bottom + (len(sections) - 1 - row) * (frame_h + gap)
    ax = fig.add_axes([x0 / width, y0 / height, frame_w / width, frame_h / height])

    # Interpolate log-resistivity from the unstructured inversion mesh onto a regular
    # plotting grid.
    distance_grid = np.linspace(0.0, span, int(round(span)) + 1)
    depth_grid = np.linspace(0.0, depth_max, 121)
    D, Z = np.meshgrid(distance_grid, depth_grid)
    log_model = np.log10(np.clip(model, 1.0, 1.0e4))
    regular = griddata((distance_m, depth_m), log_model, (D, Z), method='linear')
    edge_fill = griddata((distance_m, depth_m), log_model, (D, Z), method='nearest')
    regular = np.where(np.isfinite(regular), regular, edge_fill)
    image = ax.pcolormesh(D, Z, 10.0 ** regular, shading='auto',
                          cmap=RES_CMAP, norm=ERT_NORM, rasterized=True)
    ax.plot([0, span], [0, 0], color='#151515', lw=1.1, zorder=5)
    ax.set_xlim(0, span); ax.set_ylim(depth_max, 0)
    ax.set_aspect('equal', adjustable='box')
    ax.set_ylabel('Depth (m)')
    if row == len(sections) - 1:
        ax.set_xlabel('Distance (m)')
    else:
        ax.set_xticklabels([])
    ax.text(0.0, 1.035, DISPLAY_NAMES[site], transform=ax.transAxes,
            ha='left', va='bottom', fontsize=12, fontweight='bold')
    ax.xaxis.set_major_locator(MaxNLocator(nbins=5))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=3))
    bold_axes(ax)

cax = fig.add_axes([(left + frame_w_max + 0.20) / width,
                    bottom / height, 0.27 / width,
                    (len(sections) * frame_h + (len(sections) - 1) * gap) / height])
cb = fig.colorbar(ScalarMappable(norm=ERT_NORM, cmap=RES_CMAP), cax=cax)
cb.set_ticks(ERT_TICKS)
cb.set_ticklabels(ERT_TICKLABELS)
cb.set_label(r'Resistivity ($\Omega$ m)', fontsize=14, fontweight='bold')
cb.ax.minorticks_off()
cb.ax.tick_params(labelsize=11, width=1.2, length=5)
for label in cb.ax.get_yticklabels():
    label.set_fontweight('bold')
save_png(fig, 'Fig6b_ERT_sections.png', tight=False)
print([(DISPLAY_NAMES[s], span) for s, span, *_ in sections])


## Prepare A–A′ and GX-2 locations

In [ ]:
# -- the trace of panel d: AEM flight line L20031 where Ellis CW1 and GX-2 meet it
import sys
sys.path.insert(0, str(ROOT / 'scripts'))
from case2_cedar_rapids import _site_positions

SECTION_LINE = 'L20031'
SECTION_WINDOW_M = (1100.0, 2000.0)   # distance along the flight line
GX2_XY = np.array([604814.0, 4650366.0])  # USGS site 415954091440401
GX2_FINE_TO_COARSE_M = 3.0             # the log's fine-to-coarse change
GX2_LIMESTONE_M = 16.0                 # "shaley limestone at 55 feet", casing 0.73 m above ground

all_names = np.asarray(aem['sounding_ids'])
fit_line = np.array([str(n).split('-')[0] for n in all_names]) == SECTION_LINE
line_take = np.flatnonzero(fit_line)
line_take = line_take[np.argsort([float(str(all_names[i]).split('-')[-1]) for i in line_take])]
line_xy_all = np.asarray(aem['receiver_locations_m'][line_take, :2], float)
line_s_all = np.r_[0.0, np.cumsum(np.linalg.norm(np.diff(line_xy_all, axis=0), axis=1))]
in_window = (line_s_all >= SECTION_WINDOW_M[0]) & (line_s_all <= SECTION_WINDOW_M[1])
line_fits = np.asarray(aem['chi_squared'], float)[line_take] <= 2.0
section_take = line_take[in_window & line_fits]
section_s = line_s_all[in_window & line_fits] - SECTION_WINDOW_M[0]
section_xy = line_xy_all[in_window & line_fits]
trace_xy = line_xy_all[in_window]

def along_section(points):
    # Distance along the flight line of the nearest point on it, and how far off it.
    distance, index = cKDTree(line_xy_all).query(points)
    return line_s_all[index] - SECTION_WINDOW_M[0], distance

print(f'Section {SECTION_LINE}: {section_take.size} fitting soundings over '
      f'{SECTION_WINDOW_M[1] - SECTION_WINDOW_M[0]:.0f} m')


In [ ]:
FIT_BLUE = '#00A3E0'


## c | Borehole GX-2 and nearby AEM

In [ ]:
# Panel c: GX-2's log beside the AEM beneath it.
aem_rho = 1.0 / np.asarray(aem['conductivity_s_m'], float)
aem_tops = np.asarray(aem['layer_top_depth_m'], float)
aem_mid = np.r_[(aem_tops[:-1] + aem_tops[1:]) / 2, aem_tops[-1] + (aem_tops[-1] - aem_tops[-2]) / 2]
fitting = np.asarray(aem['chi_squared'], float) <= 2.0
aem_sites = np.asarray(aem['receiver_locations_m'], float)[:, :2]
ours_index = np.flatnonzero(fitting)[cKDTree(aem_sites[fitting]).query(GX2_XY)[1]]
ours_distance = float(np.hypot(*(aem_sites[ours_index] - GX2_XY)))

header = (ROOT / 'data/case2_Cedarrapids/aem/inversion_EM1DFM.csv').open(encoding='utf-8').readline().strip().split(',')
contractor_columns = ['X_NAD83UTM15N_M', 'Y_NAD83UTM15N_M', 'DOI'] + [f'DEP_28_TOP_NEG[{i}]' for i in range(28)] \
          + [f'DEP_28_BOT_NEG[{i}]' for i in range(28)] + [f'RES2_FINAL[{i}]' for i in range(28)]
contractor = np.loadtxt(ROOT / 'data/case2_Cedarrapids/aem/inversion_EM1DFM.csv', delimiter=',', skiprows=1,
                        usecols=[header.index(c) for c in contractor_columns])
near = contractor[np.hypot(contractor[:, 0] - GX2_XY[0], contractor[:, 1] - GX2_XY[1]) < 400]
row = near[np.argmin(np.hypot(near[:, 0] - GX2_XY[0], near[:, 1] - GX2_XY[1]))]
their_distance = float(np.hypot(row[0] - GX2_XY[0], row[1] - GX2_XY[1]))
their_mid = -(row[3:31] + row[31:59]) / 2
their_rho = row[59:87]
valid = (their_rho > 0) & (their_mid <= row[2])     # -99999 below their own DOI (F031)
print(f'nearest fitting sounding of ours {ours_distance:.0f} m from GX-2; contractor {their_distance:.0f} m, DOI {row[2]:.0f} m')

fig = plt.figure(figsize=(4.9, 3.7))
log_ax = fig.add_axes([0.14, 0.17, 0.09, 0.75])
ax = fig.add_axes([0.27, 0.17, 0.68, 0.75], sharey=log_ax)
LITHOLOGY = [(0.0, GX2_FINE_TO_COARSE_M, '#C9B79C', 'fine'), (GX2_FINE_TO_COARSE_M, GX2_LIMESTONE_M, '#EACB7A', 'sand &\ngravel'),
             (GX2_LIMESTONE_M, 19.0, '#9FB3C8', 'lime-\nstone')]
for top, bottom, colour, name in LITHOLOGY:
    log_ax.add_patch(mpl.patches.Rectangle((0, top), 1, bottom - top, facecolor=colour, edgecolor='#595959', lw=0.8))
log_ax.set_xlim(0, 1); log_ax.set_xticks([])
log_ax.set_ylim(60, 0)
log_ax.set_ylabel('Depth (m)')
log_ax.set_title('GX-2', fontsize=11, fontweight='bold', pad=4)
ax.plot(aem_rho[ours_index], aem_mid, color=FIT_BLUE, lw=2.0, label='AEM', zorder=3)
for depth, colour in ((GX2_FINE_TO_COARSE_M, '#8C6D46'), (GX2_LIMESTONE_M, '#3E5A73')):
    ax.axhline(depth, color=colour, lw=1.1, zorder=1)
    log_ax.axhline(depth, color=colour, lw=1.1)
peak = int(np.argmax(np.where(aem_mid <= 60, aem_rho[ours_index], -np.inf)))
ax.scatter(aem_rho[ours_index][peak], aem_mid[peak], s=38, color=FIT_BLUE, edgecolor='white', zorder=4)
ax.text(390, GX2_LIMESTONE_M + 1.0, 'limestone in the well', ha='right', va='top',
        fontsize=10, fontweight='bold', color='#3E5A73')
ax.text(390, GX2_FINE_TO_COARSE_M + 1.0, 'fine to coarse', ha='right', va='top',
        fontsize=10, fontweight='bold', color='#8C6D46')
ax.set_xscale('log')
ax.set_xlim(40, 400)
ax.set_xticks([50, 100, 200, 400]); ax.set_xticklabels(['50', '100', '200', '400'])
ax.xaxis.set_minor_formatter(mpl.ticker.NullFormatter())
ax.set_xlabel(r'Resistivity ($\Omega$ m)')
plt.setp(ax.get_yticklabels(), visible=False)
for side in ('top', 'right'):
    ax.spines[side].set_visible(False)
bold_axes(ax); bold_axes(log_ax)
save_png(fig, 'Fig6c_well_GX2.png')

## d | Geoelectric model

In [ ]:
fig = plt.figure(figsize=(10.2, 7.6), facecolor='white')
ax = fig.add_subplot(111, projection='3d')
style_3d(ax)

sx, sy, sz = slice(None, None, 2), slice(None), slice(None, None, 2)
xv, yv, zv = xkm[sx], ykm[sy], zkm[sz]
# Each displayed voxel merges two original cells in E and depth.
xve, yve = xedges[::2], yedges
zve = zedges[::2] if (zedges.size - 1) % 2 == 0 else np.r_[zedges[:-1:2], zedges[-1]]
unit_v = unit[sx, sy, sz]
XC, YC = np.meshgrid(xv, yv, indexing='ij')
shallow_window = (XC >= xcut) & (YC <= ycut)
filled = ~((unit_v == 1) & shallow_window[:, :, None])
face = UNIT_COLORS[unit_v].copy()
Xe, Ye, Ze = np.meshgrid(xve, yve, zve, indexing='ij')
ax.voxels(Xe, Ye, Ze, filled, facecolors=face,
          edgecolor='none', linewidth=0, shade=False)

ax.set_xlim(xedges[0], xedges[-1]); ax.set_ylim(yedges[0], yedges[-1]); ax.set_zlim(zedges[-1], 0)

# Match panel a's black native box edges.
box_color, box_lw = '#111111', 1.15
for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
    axis.pane.set_edgecolor(box_color)
    axis.pane.set_linewidth(box_lw)
    axis.pane.set_alpha(1.0)
unit_names = ['Shallow geoelectric layer', 'High-resistivity zone',
              'Low-resistivity zone', 'Below AEM DOI (uninterpreted)']
legend_order = (1, 2, 3, 0)
handles = [Patch(facecolor=UNIT_COLORS[i], edgecolor='none', label=name)
           for i, name in zip(legend_order, unit_names)]
ax.legend(handles=handles, loc='upper left', bbox_to_anchor=(0.01, 0.98), fontsize=10)
fig.subplots_adjust(left=0.00, right=0.95, bottom=0.02, top=0.99)

# Draw only the visible perimeter in screen space, above opaque voxels.
from mpl_toolkits.mplot3d import proj3d
fig.canvas.draw()
x0, x1 = xve[0], xve[-1]
y0, y1 = yve[0], yve[-1]
z0, z1 = zve[0], zedges[-1]
window_x_index = np.flatnonzero(xv >= xcut)[0]
window_y_index = np.flatnonzero(yv <= ycut)[-1]
window_x0 = xve[window_x_index]
window_y1 = yve[window_y_index + 1]
front_right_top = zve[np.flatnonzero(filled[-1, 0, :])[0]]
visible_box_edges = [
    ((x0, y0, z0), (window_x0, y0, z0)),
    ((window_x0, y0, z0), (window_x0, window_y1, z0)),
    ((window_x0, window_y1, z0), (x1, window_y1, z0)),
    ((x1, window_y1, z0), (x1, y1, z0)),
    ((x1, y1, z0), (x0, y1, z0)),
    ((x0, y1, z0), (x0, y0, z0)),
    ((x0, y0, z0), (x0, y0, z1)),
    ((x1, y0, front_right_top), (x1, y0, z1)),
]
# Follow the actual stepped top of the exposed model along both open sides.
front_open_edge = [(window_x0, y0, z0)]
for i in range(window_x_index, len(xv)):
    surface_z = zve[np.flatnonzero(filled[i, 0, :])[0]]
    front_open_edge.extend([(xve[i], y0, surface_z),
                            (xve[i + 1], y0, surface_z)])
right_open_edge = [(x1, y0, front_right_top)]
for j in range(window_y_index + 1):
    surface_z = zve[np.flatnonzero(filled[-1, j, :])[0]]
    right_open_edge.extend([(x1, yve[j], surface_z),
                            (x1, yve[j + 1], surface_z)])
right_open_edge.append((x1, window_y1, z0))
projection = ax.get_proj()
for points in [[start, end] for start, end in visible_box_edges] + [front_open_edge, right_open_edge]:
    points = np.asarray(points)
    px, py, _ = proj3d.proj_transform(
        points[:, 0], points[:, 1], points[:, 2], projection)
    pixels = ax.transData.transform(np.column_stack([px, py]))
    figure_xy = fig.transFigure.inverted().transform(pixels)
    fig.add_artist(Line2D(figure_xy[:, 0], figure_xy[:, 1],
                          transform=fig.transFigure, color=box_color,
                          lw=box_lw, zorder=1000, clip_on=False))
# The trace of panel d, drawn in screen space above the opaque voxels.
tx, ty = trace_xy[:, 0] / 1000.0, trace_xy[:, 1] / 1000.0
px, py, _ = proj3d.proj_transform(tx, ty, np.zeros_like(tx), projection)
figure_xy = fig.transFigure.inverted().transform(ax.transData.transform(np.column_stack([px, py])))
fig.add_artist(Line2D(figure_xy[:, 0], figure_xy[:, 1], transform=fig.transFigure,
                      color='#111111', lw=2.4, zorder=1001, solid_capstyle='round'))
for end, name, dx, dy, ha, va in ((0, 'A', 0.0, 0.018, 'center', 'bottom'),
                                  (-1, "A'", 0.014, -0.008, 'left', 'top')):
    fig.text(figure_xy[end, 0] + dx, figure_xy[end, 1] + dy, name, transform=fig.transFigure,
             ha=ha, va=va, fontsize=14, fontweight='bold', zorder=1002)
save_png(fig, 'Fig6d_interpreted_model_3D.png')

## e | Section A–A′ and parameter stability

In [ ]:
SECTION_NORM = ERT_NORM                      # panel b's scale; panel e carries no colour bar of its own
COVERAGE_BAR_DECADES = 2.0
AEM_REACH_FOR_PROFILES_M = 300.0
ERT_COLOR, AEM_COLOR = '#C8553D', '#2A5C8A'
model_floor_m = zedges[-1] * 1000.0

# ---------------------------------------------------------------- panel d
from scipy.interpolate import RegularGridInterpolator
import case2_cedar_rapids
from case2_cedar_rapids import _ground_along
# The case script names its data relative to the repository; this notebook runs from examples/.
if not case2_cedar_rapids.AEM.is_absolute():
    case2_cedar_rapids.AEM = ROOT / case2_cedar_rapids.AEM

column_tree = cKDTree(columns)
section_columns = column_tree.query(section_xy)[1]
section_doi = doi.reshape(-1)[section_columns]
section_rise = cover.reshape(-1)[section_columns]
section_rho = np.divide(1.0, np.asarray(aem['conductivity_s_m'], float)[section_take])
s_edges = cell_edges(section_s)

# One terrain for everything on the section: the airborne DTM at the nearest reading.
section_ground = _ground_along(section_xy)[0]
x_display = np.arange(s_edges[0], s_edges[-1] + 0.01, 2.0)
ground_display = np.interp(x_display, section_s, section_ground)
doi_display = np.interp(x_display, section_s, np.minimum(section_doi, model_floor_m))

# Display interpolation only: log10 resistivity, bilinear between neighbouring soundings
# and between layer centres.
layer_bottom = np.r_[aem_depth_top[1:], 2 * aem_depth_top[-1] - aem_depth_top[-2]]
layer_centre = (aem_depth_top + layer_bottom) / 2
log_section = RegularGridInterpolator((section_s, layer_centre), np.log10(section_rho),
                                      bounds_error=False, fill_value=None)
SECTION_DEPTH_M = 60.0                       # panel e is read to 60 m
elevation_display = np.arange(ground_display.max() - SECTION_DEPTH_M - 1.0, ground_display.max() + 0.51, 0.5)
X, E = np.meshgrid(x_display, elevation_display)
D = ground_display[None, :] - E                      # depth below the ground of each column
inside = (D >= 0.0) & (D <= doi_display[None, :])
query = np.column_stack([X.ravel(), np.clip(D, layer_centre[0], layer_centre[-1]).ravel()])
curtain = np.where(inside, 10.0 ** log_section(query).reshape(X.shape), np.nan)

# The Ellis CW1 dipole-dipole model, on the flight line's distance axis and hung on its ground.
ellis_run = CASE / 'runs' / STATE['ert_runs']['Ellis_CW1/dipole-dipole']
ellis_item = next(r for r in STATE['ert_imports'] if r['site'] == 'Ellis_CW1' and r['array'] == 'dipole-dipole')
ellis_along = np.asarray(np.load(ROOT / ellis_item['electrodes_path']), float)[:, 0]
ellis_xy = _site_positions('Ellis_CW1', ellis_along)
ellis_s, ellis_offset = along_section(ellis_xy)
ellis_ground = _ground_along(ellis_xy)[0]
to_section = np.polyfit(ellis_along, ellis_s, 1)       # linear: the line is straight
ellis_mesh = np.load(ellis_run / 'mesh_geometry.npz')['cell_centers_m']
ellis_model = np.asarray(np.load(ellis_run / 'recovered_model.npy'), float)
ellis_cov = np.asarray(np.load(ellis_run / 'model_coverage.npy'), float)
ert_along = np.linspace(ellis_along.min(), ellis_along.max(), 276)
ert_depth = np.linspace(0.0, 30.0, 121)
EA, ED = np.meshgrid(ert_along, ert_depth)
points = (ellis_mesh[:, 0], -ellis_mesh[:, 1])
log_rho = griddata(points, np.log10(ellis_model), (EA, ED), method='linear')
log_rho = np.where(np.isfinite(log_rho), log_rho, griddata(points, np.log10(ellis_model), (EA, ED), method='nearest'))
drop = griddata(points, ellis_cov.max() - ellis_cov, (EA, ED), method='linear')
drop = np.where(np.isfinite(drop), drop, griddata(points, ellis_cov.max() - ellis_cov, (EA, ED), method='nearest'))
ert_image = np.ma.masked_where(drop > COVERAGE_BAR_DECADES, 10.0 ** log_rho)
SEE_THROUGH = RES_CMAP.copy()
SEE_THROUGH.set_bad((0, 0, 0, 0))            # masked ERT shows the AEM beneath
ert_s = np.polyval(to_section, ert_along)
ert_ground = np.interp(ert_along, ellis_along, ellis_ground)
ES = np.broadcast_to(ert_s, EA.shape)
EE = ert_ground[None, :] - ED

gx2_s, gx2_offset = along_section(GX2_XY[None, :])
gx2_s, gx2_offset = float(gx2_s[0]), float(gx2_offset[0])
gx2_ground = float(_ground_along(GX2_XY[None, :])[0][0])

frame_w, frame_h = 7.2, 2.55
left, right, bottom, top = 1.05, 0.30, 1.10, 0.62
width, height = left + frame_w + right, bottom + frame_h + top
fig = plt.figure(figsize=(width, height), facecolor='white')
ax = fig.add_axes([left / width, bottom / height, frame_w / width, frame_h / height])
ax.pcolormesh(x_display, elevation_display, curtain, cmap=RES_CMAP, norm=SECTION_NORM,
              shading='nearest', rasterized=True)
ax.pcolormesh(ES, EE, ert_image, cmap=SEE_THROUGH, norm=SECTION_NORM,
              shading='nearest', rasterized=True, zorder=3)
ax.plot(np.r_[ert_s, ert_s[::-1], ert_s[0]],
        np.r_[ert_ground, ert_ground[::-1] - 30.0, ert_ground[0]], color='#111111', lw=1.0, zorder=4)
ax.plot(x_display, ground_display, color='#3A3A3A', lw=1.1, zorder=5)
ax.plot(section_s, section_ground - section_rise, color='#111111', lw=1.8, zorder=6)

ax.text(ert_s.mean(), np.interp(ert_s.mean(), x_display, ground_display) + 2.0, 'Ellis CW1 (ERT)',
        ha='center', va='bottom', fontsize=11, fontweight='bold')
for x, name in ((s_edges[0], 'A'), (s_edges[-1], "A'")):
    ax.text(x, 1.03, name, transform=ax.get_xaxis_transform(), ha='center', va='bottom',
            fontsize=14, fontweight='bold')
ax.set_xlim(s_edges[0], s_edges[-1])
ax.set_ylim(ground_display.max() - SECTION_DEPTH_M, ground_display.max() + 9.0)
ax.set_xlabel('Distance (m)')
ax.set_ylabel('Depth (m)')
ax.xaxis.set_major_locator(MaxNLocator(nbins=6))
# Drawn on elevation, so the ground keeps its relief; labelled as depth below the
# highest ground on the section, which the relief departs from by 1.4 m at most.
depth_datum = ground_display.max()
ax.set_yticks(depth_datum - np.array([0.0, 20.0, 40.0, 60.0]))
ax.yaxis.set_major_formatter(FuncFormatter(lambda value, _: f'{depth_datum - value:.0f}'))
bold_axes(ax)
handles = [Line2D([], [], color='#111111', lw=1.8, label='Resistivity-rise surface'),
           Patch(facecolor='none', edgecolor='#111111', lw=1.0, label='ERT')]
ax.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, -0.30),
          ncol=len(handles), fontsize=10, handlelength=2.4)
save_png(fig, 'Fig6e_section_L20031.png', tight=False)
print(f'Ellis CW1 electrodes {ellis_offset.min():.0f}-{ellis_offset.max():.0f} m off the line')
print(f'ground along the section {ground_display.min():.1f}-{ground_display.max():.1f} m; '
      f'Ellis CW1 {ellis_ground.min():.1f}-{ellis_ground.max():.1f} m; GX-2 collar {gx2_ground:.1f} m')


In [ ]:
# Panel e: label stability with depth, per baseline unit.
sensitivity = np.load(ROOT / STATE['model_sensitivity_path'])
by_setting = sensitivity['unit_by_setting']
geo_i = np.load(GEO_MODEL, allow_pickle=False)
unit_i = geo_i['unit']
depth_i = geo_i['ground_elevation_m'] - geo_i['cell_centres_m'][:, 2]
stable = (by_setting == unit_i[None, :]).all(axis=0)
print(f'cells that keep their label under all 27: {stable.mean():.1%}; change at least once {1 - stable.mean():.1%}')
bands = np.arange(0.0, 99.0, 4.0)
centres_i = bands[:-1] + 2.0
fig, ax = plt.subplots(figsize=(3.2, 4.6))
names = [str(n) for n in geo_i['unit_names']]
for index, name in enumerate(names):
    if name.startswith('below'):
        continue
    share = []
    for top, bottom in zip(bands[:-1], bands[1:]):
        mask = (unit_i == index) & (depth_i >= top) & (depth_i < bottom)
        share.append(stable[mask].mean() if mask.sum() >= 30 else np.nan)
    # Panel d's names, so the two panels call a unit the same thing.
    ax.plot(share, centres_i, color=UNIT_COLORS[index], lw=2.2, label=dict(zip(legend_order, unit_names))[index])
overall = [stable[(depth_i >= t) & (depth_i < b)].mean() for t, b in zip(bands[:-1], bands[1:])]
ax.plot(overall, centres_i, color='#1F1F1F', lw=1.4, ls='--', label='all cells')
ax.set_xlim(0, 1.0)
ax.set_ylim(98, 0)
ax.set_xlabel('Label stability')
ax.set_ylabel('Depth (m)')
ax.legend(loc='lower center', bbox_to_anchor=(0.42, 1.0), ncol=1, fontsize=9.5, prop={'weight': 'bold', 'size': 9.5})
for side in ('top', 'right'):
    ax.spines[side].set_visible(False)
bold_axes(ax)
save_png(fig, 'Fig6e_label_stability.png')

## Export check

Every listed panel must exist and have nonzero dimensions.

In [ ]:
from PIL import Image
expected = ['Fig6a_AEM_quasi3D.png', 'Fig6b_ERT_sections.png', 'Fig6c_well_GX2.png', 'Fig6d_interpreted_model_3D.png', 'Fig6e_section_L20031.png', 'Fig6e_label_stability.png']
for name in expected:
    path = OUT / name
    if not path.is_file():
        raise FileNotFoundError(path)
    with Image.open(path) as panel:
        assert panel.width > 0 and panel.height > 0, path
print(f'Figure panels checked: {len(expected)} in {OUT}')
